# Llama fine-tuning script

In [ ]:
# 02_train_llama_lora_or_qlora.py
# Llama fine-tuning stage only.
#
# This script:
#   1. Loads existing RAG-grounded SFT JSONL files.
#   2. Loads meta-llama/Llama-3.1-8B-Instruct.
#   3. Uses CUDA QLoRA if CUDA is available.
#   4. Uses CPU LoRA fallback if CUDA is not available.
#   5. Saves the adapter.
#
# It does NOT rebuild FAISS, embeddings, train/validation split, or RAG SFT records.
#
# Install:
#   pip install -U torch transformers datasets accelerate peft trl bitsandbytes sentencepiece protobuf safetensors huggingface_hub

import gc
import json
import os
import random
import getpass
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

import torch
from datasets import Dataset
from huggingface_hub import get_token, login
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.trainer_utils import get_last_checkpoint
from trl import SFTConfig, SFTTrainer


@dataclass
class Config:
    output_dir: str = r"C:\Users\XXXXXXXXX\Project_RAG\outputs"

    generator_model_id: str = "meta-llama/Llama-3.1-8B-Instruct"

    # If True, ask for token every run. If False, use HF_TOKEN or cached Hugging Face login first.
    force_token_prompt: bool = False

    seed: int = 42

    # CUDA QLoRA settings.
    max_seq_length: int = 3072
    num_train_epochs: float = 1.0
    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.03
    weight_decay: float = 0.0
    logging_steps: int = 25
    eval_steps: int = 250
    save_steps: int = 500
    save_total_limit: int = 2
    gradient_checkpointing: bool = True

    # CPU LoRA fallback settings.
    # CPU training is very slow. Keep 800/100 unless you intentionally want a full CPU run.
    cpu_train_record_limit: Optional[int] = 800
    cpu_eval_record_limit: int = 100
    cpu_max_seq_length: int = 1024
    cpu_max_steps: Optional[int] = None

    # CPU dtype. "float16" uses less RAM. If it errors, try "float32".
    cpu_torch_dtype: str = "float16"
    cpu_allow_float32_fallback: bool = True

    # LoRA adapters for Llama-family causal LMs.
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_target_modules: Tuple[str, ...] = (
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    )


CFG = Config()


def ask_for_hf_token(cfg: Config) -> str:
    cached = os.environ.get("HF_TOKEN") or get_token()

    if cfg.force_token_prompt or not cached:
        token = getpass.getpass("Paste your Hugging Face token. It will not be shown: ").strip()

        if not token:
            raise RuntimeError("No Hugging Face token was entered.")

        os.environ["HF_TOKEN"] = token
        login(token=token, add_to_git_credential=False)

        return token

    os.environ["HF_TOKEN"] = cached
    return cached


def dtype_from_name(name: str) -> torch.dtype:
    name = name.lower().strip()

    if name in {"float16", "fp16", "half"}:
        return torch.float16

    if name in {"bfloat16", "bf16"}:
        return torch.bfloat16

    if name in {"float32", "fp32"}:
        return torch.float32

    raise ValueError(f"Unsupported dtype name: {name}")


def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_dirs(cfg: Config) -> Dict[str, Path]:
    out = Path(cfg.output_dir)

    paths = {
        "output": out,
        "sft_data": out / "sft_data",
        "adapter": out / "llama31_8b_rag_lora_or_qlora_adapter",
        "trainer_runs": out / "trainer_runs_lora_or_qlora",
    }

    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)

    return paths


def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                rows.append(json.loads(line))

    return rows


def load_sft_records(paths: Dict[str, Path]) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    train_path = paths["sft_data"] / "train_sft_prompt_completion.jsonl"
    val_path = paths["sft_data"] / "validation_sft_prompt_completion.jsonl"

    if not train_path.exists():
        raise FileNotFoundError(f"Missing train SFT file: {train_path}. Run 01_build_rag_training_data.py first.")

    if not val_path.exists():
        raise FileNotFoundError(f"Missing validation SFT file: {val_path}. Run 01_build_rag_training_data.py first.")

    train_records = read_jsonl(train_path)
    val_records = read_jsonl(val_path)

    return train_records, val_records


def load_training_tokenizer(cfg: Config, token: str) -> AutoTokenizer:
    tokenizer = AutoTokenizer.from_pretrained(
        cfg.generator_model_id,
        token=token,
        use_fast=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"

    return tokenizer


def load_training_model(cfg: Config, token: str) -> Tuple[AutoModelForCausalLM, str]:
    if torch.cuda.is_available():
        bf16 = torch.cuda.is_bf16_supported()
        compute_dtype = torch.bfloat16 if bf16 else torch.float16

        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=True,
        )

        model = AutoModelForCausalLM.from_pretrained(
            cfg.generator_model_id,
            token=token,
            quantization_config=quant_config,
            torch_dtype=compute_dtype,
            device_map="auto",
            low_cpu_mem_usage=True,
        )

        model.config.use_cache = False

        model = prepare_model_for_kbit_training(
            model,
            use_gradient_checkpointing=cfg.gradient_checkpointing,
        )

        return model, "cuda_qlora"

    cpu_dtype = dtype_from_name(cfg.cpu_torch_dtype)

    try:
        model = AutoModelForCausalLM.from_pretrained(
            cfg.generator_model_id,
            token=token,
            torch_dtype=cpu_dtype,
            low_cpu_mem_usage=True,
        )
    except Exception as exc:
        if cfg.cpu_allow_float32_fallback and cpu_dtype != torch.float32:
            print(f"CPU model load with {cfg.cpu_torch_dtype} failed. Retrying with float32.")
            print(f"Original error: {exc}")

            model = AutoModelForCausalLM.from_pretrained(
                cfg.generator_model_id,
                token=token,
                torch_dtype=torch.float32,
                low_cpu_mem_usage=True,
            )
        else:
            raise

    model.config.use_cache = False

    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()

    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()

    return model, "cpu_lora"


def make_sft_config(cfg: Config, paths: Dict[str, Path], training_mode: str) -> SFTConfig:
    is_cuda = training_mode == "cuda_qlora"

    kwargs = {
        "output_dir": str(paths["trainer_runs"]),
        "max_length": cfg.max_seq_length if is_cuda else cfg.cpu_max_seq_length,
        "packing": False,
        "completion_only_loss": True,
        "num_train_epochs": cfg.num_train_epochs,
        "per_device_train_batch_size": cfg.per_device_train_batch_size if is_cuda else 1,
        "per_device_eval_batch_size": cfg.per_device_eval_batch_size if is_cuda else 1,
        "gradient_accumulation_steps": cfg.gradient_accumulation_steps if is_cuda else 1,
        "learning_rate": cfg.learning_rate,
        "warmup_ratio": cfg.warmup_ratio,
        "weight_decay": cfg.weight_decay,
        "optim": "paged_adamw_8bit" if is_cuda else "adamw_torch",
        "bf16": bool(is_cuda and torch.cuda.is_bf16_supported()),
        "fp16": bool(is_cuda and not torch.cuda.is_bf16_supported()),
        "gradient_checkpointing": cfg.gradient_checkpointing,
        "logging_steps": cfg.logging_steps,
        "save_strategy": "steps",
        "save_steps": cfg.save_steps,
        "save_total_limit": cfg.save_total_limit,
        "report_to": "none",
        "seed": cfg.seed,
        "data_seed": cfg.seed,
    }

    if is_cuda:
        kwargs["eval_strategy"] = "steps"
        kwargs["eval_steps"] = cfg.eval_steps
    else:
        kwargs["eval_strategy"] = "no"

        if cfg.cpu_max_steps is not None:
            kwargs["max_steps"] = cfg.cpu_max_steps

    for _ in range(8):
        try:
            return SFTConfig(**kwargs)
        except TypeError as exc:
            msg = str(exc)
            handled = False

            if "completion_only_loss" in msg and "completion_only_loss" in kwargs:
                kwargs.pop("completion_only_loss")
                handled = True

            if "max_length" in msg and "max_length" in kwargs:
                kwargs["max_seq_length"] = kwargs.pop("max_length")
                handled = True

            if "eval_strategy" in msg and "eval_strategy" in kwargs:
                kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
                handled = True

            if not handled:
                raise

    return SFTConfig(**kwargs)


def train_lora_or_qlora(
    train_records: List[Dict[str, Any]],
    val_records: List[Dict[str, Any]],
    cfg: Config,
    paths: Dict[str, Path],
) -> None:
    token = ask_for_hf_token(cfg)

    tokenizer = load_training_tokenizer(cfg, token)
    model, training_mode = load_training_model(cfg, token)

    print(f"Training mode: {training_mode}")

    if training_mode == "cpu_lora" and cfg.cpu_train_record_limit is not None:
        train_used = train_records[: cfg.cpu_train_record_limit]
    else:
        train_used = train_records

    if training_mode == "cpu_lora":
        val_used = val_records[: cfg.cpu_eval_record_limit]
    else:
        val_used = val_records

    print(f"Train records available: {len(train_records):,}")
    print(f"Validation records available: {len(val_records):,}")
    print(f"Train records used by Trainer: {len(train_used):,}")
    print(f"Validation records used by Trainer: {len(val_used):,}")

    train_dataset = Dataset.from_list(train_used)
    val_dataset = Dataset.from_list(val_used)

    lora_config = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=list(cfg.lora_target_modules),
    )

    training_args = make_sft_config(cfg, paths, training_mode)

    trainer_kwargs = {
        "model": model,
        "args": training_args,
        "train_dataset": train_dataset,
        "eval_dataset": val_dataset,
        "peft_config": lora_config,
    }

    try:
        trainer = SFTTrainer(
            **trainer_kwargs,
            processing_class=tokenizer,
        )
    except TypeError as exc:
        if "processing_class" not in str(exc):
            raise

        trainer = SFTTrainer(
            **trainer_kwargs,
            tokenizer=tokenizer,
        )

    last_checkpoint = get_last_checkpoint(str(paths["trainer_runs"])) if paths["trainer_runs"].exists() else None

    if last_checkpoint:
        print(f"Resuming Trainer checkpoint: {last_checkpoint}")
        trainer.train(resume_from_checkpoint=last_checkpoint)
    else:
        print("No Trainer checkpoint found. Starting adapter training from the beginning.")
        trainer.train()

    trainer.save_model(str(paths["adapter"]))
    tokenizer.save_pretrained(str(paths["adapter"]))

    manifest = {
        "config": asdict(cfg),
        "generator_model_id": cfg.generator_model_id,
        "training_mode": training_mode,
        "adapter_dir": str(paths["adapter"]),
        "trainer_runs_dir": str(paths["trainer_runs"]),
        "train_records_total_available": len(train_records),
        "validation_records_total_available": len(val_records),
        "train_records_used": len(train_used),
        "validation_records_for_trainer": len(val_used),
        "cuda_available": torch.cuda.is_available(),
    }

    with open(paths["output"] / "training_manifest_lora_or_qlora.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    print("\nTraining complete.")
    print(f"Adapter saved to: {paths['adapter']}")


def main() -> None:
    set_seed(CFG.seed)
    paths = ensure_dirs(CFG)

    with open(paths["output"] / "config_train_lora_or_qlora.json", "w", encoding="utf-8") as f:
        json.dump(asdict(CFG), f, ensure_ascii=False, indent=2)

    train_records, val_records = load_sft_records(paths)

    print(f"SFT train records loaded: {len(train_records):,}")
    print(f"SFT validation records loaded: {len(val_records):,}")

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Starting Llama adapter training...")
    train_lora_or_qlora(train_records, val_records, CFG, paths)


if __name__ == "__main__":
    main()